In [1]:

# ============================================================
# SMARTBARN VISION
# Cow Detection, Tracking and Automatic Image Enhancement
# YOLO11 + ByteTrack / BoT-SORT + OpenCV
# ============================================================

from pathlib import Path
from collections import defaultdict, deque
import cv2
import numpy as np
import pandas as pd
import torch
import time
import colorsys
from ultralytics import YOLO

# ============================================================
# 1. CONFIGURATION
# ============================================================

VIDEO_PATH = Path("videos/Maa10.20210304_000000.mp4")

OUTPUT_DIR = Path("outputs/smartbarn")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO = OUTPUT_DIR / "smartbarn_tracking.mp4"
OUTPUT_CSV = OUTPUT_DIR / "tracking_results.csv"
OUTPUT_SUMMARY = OUTPUT_DIR / "tracking_summary.csv"

MODEL_NAME = "yolo11n.pt"

DEVICE = 0 if torch.cuda.is_available() else "cpu"

# Detection
CONFIDENCE = 0.10
IOU_THRESHOLD = 0.50
IMAGE_SIZE = 960
COW_CLASS = 19

# Tracking
TRACKER = "bytetrack.yaml"
# Alternative: "botsort.yaml"

# Processing
MAX_SECONDS = 10
MAX_VIDEO_SIDE = 960
FRAME_SKIP = 1

# Enhancement
AUTO_ENHANCE = True
CLAHE_CLIP = 2.0
CLAHE_GRID = (8, 8)

# Visualization
DRAW_BOXES = True
DRAW_LABELS = True
DRAW_TRAJECTORIES = True
DRAW_CENTERS = True

TRAJECTORY_LENGTH = 40

# ============================================================
# 2. LOAD MODEL
# ============================================================

print("Loading YOLO model...")

model = YOLO(MODEL_NAME)

print("Model:", MODEL_NAME)
print("Device:", DEVICE)

# ============================================================
# 3. AUTOMATIC IMAGE ANALYSIS
# ============================================================

clahe = cv2.createCLAHE(
    clipLimit=CLAHE_CLIP,
    tileGridSize=CLAHE_GRID
)

def analyze_frame(frame):
    """
    Estimate whether a frame is nearly grayscale
    and whether it has low contrast.

    OpenCV normally reads color videos as BGR.
    Grayscale videos may also be decoded as BGR.
    """

    if frame.ndim == 2:
        gray = frame
        is_grayscale = True

    elif frame.shape[2] == 1:
        gray = frame[:, :, 0]
        is_grayscale = True

    else:
        gray = cv2.cvtColor(
            frame, cv2.COLOR_BGR2GRAY
        )

        # Check similarity of color channels.
        # Small differences suggest grayscale imagery.
        b, g, r = cv2.split(frame)

        sample = (
            slice(None, None, 8),
            slice(None, None, 8)
        )

        bs = b[sample].astype(np.float32)
        gs = g[sample].astype(np.float32)
        rs = r[sample].astype(np.float32)

        color_difference = (
            np.mean(np.abs(bs - gs)) +
            np.mean(np.abs(gs - rs)) +
            np.mean(np.abs(bs - rs))
        ) / 3.0

        is_grayscale = color_difference < 5.0

    contrast = float(np.std(gray))

    return is_grayscale, contrast


# ============================================================
# 4. IMAGE ENHANCEMENT
# ============================================================

def enhance_frame(frame):
    """
    Apply CLAHE automatically when:
    - frame is nearly grayscale, or
    - frame has low contrast.

    Preserve color for ordinary RGB footage.
    """

    if frame.ndim == 2:
        frame = cv2.cvtColor(
            frame, cv2.COLOR_GRAY2BGR
        )

    elif frame.ndim == 3 and frame.shape[2] == 1:
        frame = cv2.cvtColor(
            frame, cv2.COLOR_GRAY2BGR
        )

    if not AUTO_ENHANCE:
        return frame, "Original"

    is_gray, contrast = analyze_frame(frame)

    if is_gray:
        gray = cv2.cvtColor(
            frame, cv2.COLOR_BGR2GRAY
        )

        enhanced = clahe.apply(gray)

        enhanced = cv2.cvtColor(
            enhanced, cv2.COLOR_GRAY2BGR
        )

        return enhanced, "CLAHE-Grayscale"

    elif contrast < 45:
        # Enhance luminance without destroying color.
        lab = cv2.cvtColor(
            frame, cv2.COLOR_BGR2LAB
        )

        l, a, b = cv2.split(lab)

        l = clahe.apply(l)

        enhanced = cv2.merge((l, a, b))

        enhanced = cv2.cvtColor(
            enhanced, cv2.COLOR_LAB2BGR
        )

        return enhanced, "CLAHE-Color"

    return frame, "Original"


# ============================================================
# 5. TRACK COLOR
# ============================================================

def get_track_color(track_id):
    """
    Deterministic color for each tracker ID.
    """

    hue = (track_id * 0.61803398875) % 1.0

    r, g, b = colorsys.hsv_to_rgb(
        hue, 0.75, 0.95
    )

    return (
        int(b * 255),
        int(g * 255),
        int(r * 255)
    )


# ============================================================
# 6. CUSTOM COW LABEL
# ============================================================

def draw_cow_label(
    frame,
    track_id,
    bbox,
    confidence
):
    """
    Draw a white ID label with a colored
    cow marker, similar to the reference image.
    """

    x1, y1, x2, y2 = map(int, bbox)

    color = get_track_color(track_id)

    label = f"COW-{track_id:04d}"
    score = f"{confidence:.2f}"

    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.52
    thickness = 2

    (text_w, text_h), baseline = cv2.getTextSize(
        label, font, font_scale, thickness
    )

    label_w = max(105, text_w + 40)
    label_h = 32

    # Place label near the top of the cow.
    label_x = max(
        0,
        min(x1, frame.shape[1] - label_w - 1)
    )

    label_y = max(0, y1 - label_h - 8)

    # White label background
    cv2.rectangle(
        frame,
        (label_x, label_y),
        (label_x + label_w, label_y + label_h),
        (245, 245, 245),
        -1
    )

    # Colored marker on right
    cv2.rectangle(
        frame,
        (label_x + label_w - 28, label_y),
        (label_x + label_w, label_y + label_h),
        color,
        -1
    )

    cv2.putText(
        frame,
        label,
        (label_x + 6, label_y + 22),
        font,
        font_scale,
        (25, 25, 25),
        thickness,
        cv2.LINE_AA
    )

    # Cow symbol (simple vector marker)
    marker_x = label_x + label_w - 14
    marker_y = label_y + 16

    cv2.circle(
        frame,
        (marker_x, marker_y),
        5,
        (255, 255, 255),
        2
    )

    # Confidence underneath
    cv2.putText(
        frame,
        score,
        (x1, min(frame.shape[0] - 5, y2 + 17)),
        font,
        0.42,
        color,
        1,
        cv2.LINE_AA
    )


# ============================================================
# 7. VIDEO INITIALIZATION
# ============================================================

assert VIDEO_PATH.exists(), (
    f"Video not found: {VIDEO_PATH.resolve()}"
)

cap = cv2.VideoCapture(str(VIDEO_PATH))

assert cap.isOpened(), "Cannot open video."

source_fps = cap.get(cv2.CAP_PROP_FPS)

if source_fps <= 0:
    source_fps = 20.0

original_width = int(
    cap.get(cv2.CAP_PROP_FRAME_WIDTH)
)

original_height = int(
    cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
)

scale = min(
    1.0,
    MAX_VIDEO_SIDE / max(
        original_width,
        original_height
    )
)

width = max(
    2,
    int(original_width * scale) // 2 * 2
)

height = max(
    2,
    int(original_height * scale) // 2 * 2
)

output_fps = source_fps / FRAME_SKIP

writer = cv2.VideoWriter(
    str(OUTPUT_VIDEO),
    cv2.VideoWriter_fourcc(*"mp4v"),
    output_fps,
    (width, height)
)

assert writer.isOpened(), "Video writer failed."

print("\nVideo information")
print("Original:", original_width, original_height)
print("Processing:", width, height)
print("Source FPS:", source_fps)
print("Output FPS:", output_fps)

# ============================================================
# 8. TRACKING DATA
# ============================================================

track_history = defaultdict(
    lambda: deque(maxlen=TRAJECTORY_LENGTH)
)

tracking_records = []

frame_index = 0
processed_frames = 0
max_frames = int(source_fps * MAX_SECONDS)

start_time = time.perf_counter()

enhancement_stats = defaultdict(int)

# ============================================================
# 9. MAIN VIDEO PROCESSING LOOP
# ============================================================

try:

    while frame_index < max_frames:

        success, frame = cap.read()

        if not success:
            break

        current_frame = frame_index
        frame_index += 1

        if current_frame % FRAME_SKIP != 0:
            continue

        # Resize frame
        frame = cv2.resize(
            frame,
            (width, height),
            interpolation=cv2.INTER_AREA
        )

        # Automatic enhancement
        enhanced, enhancement_type = enhance_frame(
            frame
        )

        enhancement_stats[enhancement_type] += 1

        # YOLO detection + tracking
        results = model.track(
            enhanced,
            persist=True,
            tracker=TRACKER,
            classes=[COW_CLASS],
            conf=CONFIDENCE,
            iou=IOU_THRESHOLD,
            imgsz=IMAGE_SIZE,
            device=DEVICE,
            verbose=False
        )

        result = results[0]

        # Draw on original frame rather than
        # enhanced image for natural appearance.
        annotated = frame.copy()

        visible_cows = 0

        if (
            result.boxes is not None
            and result.boxes.is_track
            and result.boxes.id is not None
        ):

            boxes = result.boxes.xyxy.cpu().numpy()

            ids = (
                result.boxes.id
                .int()
                .cpu()
                .tolist()
            )

            confidences = (
                result.boxes.conf
                .cpu()
                .tolist()
            )

            for bbox, track_id, confidence in zip(
                boxes, ids, confidences
            ):

                x1, y1, x2, y2 = bbox

                center_x = (x1 + x2) / 2
                center_y = (y1 + y2) / 2

                # Bottom-center is a useful approximate
                # ground-contact anchor in some views.
                # It is not a calibrated ground position.
                anchor_x = int(center_x)
                anchor_y = int(y2)

                color = get_track_color(track_id)

                visible_cows += 1

                # Draw bounding box
                if DRAW_BOXES:
                    cv2.rectangle(
                        annotated,
                        (int(x1), int(y1)),
                        (int(x2), int(y2)),
                        color,
                        2
                    )

                # Draw custom cow ID label
                if DRAW_LABELS:
                    draw_cow_label(
                        annotated,
                        track_id,
                        bbox,
                        confidence
                    )

                # Store trajectory
                track_history[track_id].append(
                    (anchor_x, anchor_y)
                )

                # Draw center
                if DRAW_CENTERS:
                    cv2.circle(
                        annotated,
                        (anchor_x, anchor_y),
                        4,
                        color,
                        -1
                    )

                # Draw trajectory
                if DRAW_TRAJECTORIES:

                    points = list(
                        track_history[track_id]
                    )

                    for i in range(1, len(points)):

                        cv2.line(
                            annotated,
                            points[i - 1],
                            points[i],
                            color,
                            2,
                            cv2.LINE_AA
                        )

                # Save data
                tracking_records.append({

                    "frame": current_frame,
                    "time_seconds": (
                        current_frame / source_fps
                    ),

                    "track_id": track_id,

                    "confidence": confidence,

                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,

                    "center_x": center_x,
                    "center_y": center_y,

                    "anchor_x": anchor_x,
                    "anchor_y": anchor_y,

                    "enhancement": enhancement_type
                })

        # Information overlay
        cv2.rectangle(
            annotated,
            (0, 0),
            (width, 65),
            (25, 25, 25),
            -1
        )

        cv2.putText(
            annotated,
            f"SMARTBARN VISION | Cows: {visible_cows}",
            (12, 25),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.58,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

        cv2.putText(
            annotated,
            f"Mode: {enhancement_type} | "
            f"Time: {current_frame/source_fps:.1f}s",
            (12, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (180, 220, 255),
            1,
            cv2.LINE_AA
        )

        writer.write(annotated)

        processed_frames += 1

        if processed_frames % 50 == 0:
            print(
                f"Processed {processed_frames} frames"
            )

finally:

    cap.release()
    writer.release()

# ============================================================
# 10. EXPORT TRACKING DATA
# ============================================================

columns = [
    "frame",
    "time_seconds",
    "track_id",
    "confidence",
    "x1", "y1", "x2", "y2",
    "center_x", "center_y",
    "anchor_x", "anchor_y",
    "enhancement"
]

df = pd.DataFrame(
    tracking_records,
    columns=columns
)

df.to_csv(
    OUTPUT_CSV,
    index=False
)

if not df.empty:

    summary = df.groupby(
        "track_id"
    ).agg(

        frames_detected=("frame", "count"),

        first_seen=("time_seconds", "min"),

        last_seen=("time_seconds", "max"),

        mean_confidence=("confidence", "mean")

    ).reset_index()

else:

    summary = pd.DataFrame(
        columns=[
            "track_id",
            "frames_detected",
            "first_seen",
            "last_seen",
            "mean_confidence"
        ]
    )

summary.to_csv(
    OUTPUT_SUMMARY,
    index=False
)

# ============================================================
# 11. FINAL STATISTICS
# ============================================================

elapsed = time.perf_counter() - start_time

print("\n" + "=" * 55)
print("SMARTBARN VISION COMPLETED")
print("=" * 55)

print("Processed frames:", processed_frames)
print("Total tracked detections:", len(df))

print(
    "Unique tracker IDs:",
    df["track_id"].nunique()
)

print(
    "Processing FPS:",
    round(processed_frames / elapsed, 2)
    if elapsed > 0 else 0
)

print("Enhancement usage:", dict(enhancement_stats))

print("\nOutput video:", OUTPUT_VIDEO.resolve())
print("Tracking CSV:", OUTPUT_CSV.resolve())
print("Summary CSV:", OUTPUT_SUMMARY.resolve())

display(summary.head(20))


Loading YOLO model...
Model: yolo11n.pt
Device: cpu

Video information
Original: 1944 2592
Processing: 720 960
Source FPS: 20.004876258010697
Output FPS: 20.004876258010697
Processed 50 frames
Processed 100 frames
Processed 150 frames
Processed 200 frames

SMARTBARN VISION COMPLETED
Processed frames: 200
Total tracked detections: 689
Unique tracker IDs: 4
Processing FPS: 6.24
Enhancement usage: {'CLAHE-Grayscale': 200}

Output video: D:\project-cow\outputs\smartbarn\smartbarn_tracking.mp4
Tracking CSV: D:\project-cow\outputs\smartbarn\tracking_results.csv
Summary CSV: D:\project-cow\outputs\smartbarn\tracking_summary.csv


,track_id,frames_detected,first_seen,last_seen,mean_confidence
0,1,190,0.000000,9.847599,0.457398
1,2,178,0.000000,9.747623,0.398916
2,3,179,0.000000,9.747623,0.293618
3,5,142,1.899537,9.947575,0.326120


In [5]:
%pip install imageio-ffmpeg

   ---------------------------------------- 0.0/38.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.9 MB ? eta -:--:--
   ---------------------------------------- 0.1/38.9 MB 1.1 MB/s eta 0:00:36
   ---------------------------------------- 0.1/38.9 MB 1.1 MB/s eta 0:00:36
   ---------------------------------------- 0.1/38.9 MB 726.2 kB/s eta 0:00:54
   ---------------------------------------- 0.2/38.9 MB 980.4 kB/s eta 0:00:40
   ---------------------------------------- 0.2/38.9 MB 1.0 MB/s eta 0:00:38
   ---------------------------------------- 0.2/38.9 MB 1.0 MB/s eta 0:00:38
   ---------------------------------------- 0.2/38.9 MB 1.0 MB/s eta 0:00:38
   ---------------------------------------- 0.2/38.9 MB 654.9 kB/s eta 0:01:00
   ---------------------------------------- 0.2/38.9 MB 654.9 kB/s eta 0:01:00
   ---------------------------------------- 0.3/38.9 MB 607.9 kB/s eta 0:01:04
   -------


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: D:\project-cow\cow\Scripts\python.exe -m pip install --upgrade pip


In [7]:

from pathlib import Path
from IPython.display import Video, display
import imageio_ffmpeg
import subprocess

# Original output from SmartBarn Vision
input_video = Path("outputs/smartbarn/smartbarn_tracking.mp4")

# Browser-compatible output
output_video = Path("outputs/smartbarn/smartbarn_tracking_H264.mp4")

assert input_video.exists(), f"Video not found: {input_video}"

# Get FFmpeg executable automatically
ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()

# Convert to H.264
command = [
    ffmpeg,
    "-y",
    "-i", str(input_video),
    "-c:v", "libx264",
    "-preset", "fast",
    "-crf", "23",
    "-pix_fmt", "yuv420p",
    "-movflags", "+faststart",
    str(output_video)
]

subprocess.run(command, check=True)

print("Conversion completed!")
print("Output:", output_video.resolve())
print("File size:", round(output_video.stat().st_size / 1e6, 2), "MB")

# Display inside Jupyter
display(Video(
    filename=str(output_video),
    embed=True,
    width=850,
    html_attributes="controls"
))


Conversion completed!
Output: D:\project-cow\outputs\smartbarn\smartbarn_tracking_H264.mp4
File size: 0.53 MB
